In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from rapidfuzz import process, fuzz
import pickle 

import json


In [3]:
# Dictionary of file paths
files = {
    "data_2014": "playersheets/year_totals/2014.csv",
    "data_2015": "playersheets/year_totals/2015.csv",
    "data_2016": "playersheets/year_totals/2016.csv",
    "data_2017": "playersheets/year_totals/2017.csv",
    "data_2018": "playersheets/year_totals/2018.csv",
    "data_2019": "playersheets/year_totals/2019.csv",
    "data_2020": "playersheets/year_totals/2020.csv",
    "data_2021": "playersheets/year_totals/2021.csv",
    "data_2022": "playersheets/year_totals/2022.csv",
    "data_2023": "playersheets/year_totals/2023.csv",
    "data_2024": "playersheets/year_totals/2024.csv"
}

# Load and concatenate all files
dataframes = []
for year, file_path in files.items():
    df = pd.read_csv(file_path)
    df.rename(columns={"PLAYER_NAME": "Player"}, inplace=True)
    df["Player"] = df["Player"].fillna("").astype(str).str.strip().str.upper()

    dataframes.append(df)

# Concatenate all DataFrames
player_sheets = pd.concat(dataframes, ignore_index=True)

# Save to a single CSV if needed
player_sheets.to_csv("all_years_data.csv", index=False)

# Display the shape and preview of the merged DataFrame
print(f"Merged DataFrame shape: {player_sheets.shape}")
# print(player_sheets.head())
print(type(player_sheets["Player"]))

Merged DataFrame shape: (5795, 498)
<class 'pandas.core.series.Series'>


In [4]:
files = {
    "raw_league_averages": "site_Data/avg_shooting.csv",
    "dfg": "site_Data/dfg.csv",
    "hustle": "site_Data/hustle.csv",
    "passing": "site_Data/passing.csv",
    "player_shooting": "site_Data/player_shooting.csv",
    "rim_acc_on_off": "site_Data/rim_acc.csv",
    "rim_defense": "site_Data/rimdfg.csv",
    "rim_freq_on_off": "site_Data/rimfreq.csv",
    "scoring_by_zone": "site_Data/shotzone.csv",
    "random_stats": "site_Data/wowy/player_large.csv",
    "close_6": "site_Data/player_tracking/close_6.csv",
    "catch_shoot": "site_Data/player_tracking/cs.csv",
    "drives": "site_Data/player_tracking/drives.csv",
    "passing2": "site_Data/player_tracking/passing.csv",
    "pullup": "site_Data/player_tracking/pullup.csv",
    "touches": "site_Data/player_tracking/touches.csv",
    "wide_open": "site_Data/player_tracking/wide_open.csv",
    "p_sheets": "all_years_data.csv"
}

# Read all files into DataFrames
dfs = {name: pd.read_csv(path) for name, path in files.items()}

# Ensure column consistency and standardize keys
for name, df in dfs.items():
    
    if "Name" in df.columns:
        df.rename(columns={"Name": "Player"}, inplace=True)
    if "PLAYER" in df.columns:
        df.rename(columns={"PLAYER": "Player"}, inplace=True)
    if "Player" in df.columns:
        if isinstance(df['Player'], pd.DataFrame):
            df = player_sheets
        df["Player"] = df["Player"].str.strip().str.upper()  # Standardize Player name
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")  # Ensure numeric year

# Start merging
merged_df = dfs["dfg"]  # Initialize the merged DataFrame

for name, df in dfs.items():
    if name == "rim_defense":
        df['diff_combined'] = df["Diff%"].fillna(df["DIFF%"])
        df = df.drop(columns=["Diff%", "DIFF%"])
        df.rename(columns={"diff_combined": "Diff%"}, inplace=True)
    if name == "rapm":  # Special handling for the RAPM file
        continue
        merged_df = merged_df.merge(df, on="Player", how="outer")
    elif name == "p_sheets":
        merged_df = merged_df.merge(df, left_on=["EntityId", "year"],right_on=["PLAYER_ID","year"], how="outer", suffixes=('', f'_{name}'))

    elif name=="player_shooting":
        continue
    elif name == "dfg":
        continue
    # Skip files without both "Player" and "year" columns
    elif "Player" not in df.columns or "year" not in df.columns:
        print(f"Skipping file {name} as it lacks 'Player' and/or 'year' columns.")
        continue

    elif isinstance(df['Player'], pd.DataFrame):
        df = player_sheets
    # Merge with the main DataFrame
    elif merged_df is None:
        merged_df = df
    else:
        merged_df = merged_df.merge(df, on=["Player", "year"], how="outer", suffixes=('', f'_{name}'))

merged_df.to_csv("merged_dataset.csv", index=False)

Skipping file raw_league_averages as it lacks 'Player' and/or 'year' columns.
Skipping file rim_acc_on_off as it lacks 'Player' and/or 'year' columns.
Skipping file rim_freq_on_off as it lacks 'Player' and/or 'year' columns.
Skipping file random_stats as it lacks 'Player' and/or 'year' columns.
Skipping file close_6 as it lacks 'Player' and/or 'year' columns.
Skipping file catch_shoot as it lacks 'Player' and/or 'year' columns.
Skipping file drives as it lacks 'Player' and/or 'year' columns.
Skipping file passing2 as it lacks 'Player' and/or 'year' columns.
Skipping file pullup as it lacks 'Player' and/or 'year' columns.
Skipping file touches as it lacks 'Player' and/or 'year' columns.
Skipping file wide_open as it lacks 'Player' and/or 'year' columns.


In [5]:
# Extract column names
column_names = merged_df.columns.tolist()

# Save column names to a text file
with open('column_names.txt', 'w') as file:
    for name in column_names:
        file.write(name + '\n')

print("Column names saved to 'column_names.txt'")

Column names saved to 'column_names.txt'


In [6]:
merged_df = pd.read_csv("merged_dataset.csv")

/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/4175133354.py:1: DtypeWarning: Columns (1,3,13,66,68,116,120,222,375) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv("merged_dataset.csv")


In [7]:

# Load the dataset
data = merged_df.copy()  # Replace with your file name
# Selected columns
selected_columns = [
    'Player', 'AGE','year', 'GP', 'Min', 
    'Points', 'Assists', 'Turnovers', 'FG%', 
    'High Value Assist %', 'AtRimAssists',
    'OffPoss', 'DefPoss', 'TsPct', 
    'AtRimFGA', 'AtRimAccuracy', 
    'LongMidRangeFGA', 'LongMidRangeFGM', 
    'Corner3FGA', 'Corner3FGM', 'ThreePtAssists', 
    'DFGA_rim_defense', 'DFG%_rim_defense', 'Diff%',
    'Deflections', '% Loose BallsRecovered DEF', 'ChargesDrawn', 'Contested2PT Shots',
    'on-ball-time%', 'FtPoints','NET_RATING','OREB','OnOffRtg','BLK','DREB','PF','Usage','UAFGM','Net Passes',
    'Contested 3s','Steals','FG3A','Offensive Fouls Drawn',
    'SelfOReb','DREB_CONTEST','OREB_CONTEST','OREB_UNCONTEST','DREB_UNCONTEST','RecoveredBlocks','OpponentPoints'
]
data["Net Passes"] = data["Passes"] - data["PASSES_RECEIVED"]

data["Contested 3s"] = data["tight_FG3A"]+data["very_tight_FG3A"]
# Filter the data to only include selected columns
# data = data[selected_columns]

# Ensure OffPoss and DefPoss are numeric
data['OffPoss'] = pd.to_numeric(data['OffPoss'], errors='coerce').fillna(0)
data['DefPoss'] = pd.to_numeric(data['DefPoss'], errors='coerce').fillna(0)

# Calculate total possessions
data['TotalPoss'] = data['OffPoss'] + data['DefPoss']
# Filter players with at least 10 offensive or defensive possessions
filtered_data = data[(data['OffPoss'] >= 10) & (data['DefPoss'] >= 10)]

# Reset index for the filtered dataset
filtered_data = filtered_data.reset_index(drop=True)
data = filtered_data
# Convert stats to per 100 offensive possessions
data["AFGM"] = data["FGM"] - data["UAFGM"]
offensive_stats = [
    'Points', 'Assists', 'Turnovers','FGA','FTA',
    'AtRimFGA', 'Corner3FGA', 'Corner3FGM', 
    'ThreePtAssists', 'FtPoints', 'AtRimAssists','LongMidRangeFGA','LongMidRangeFGM','OREB','UAFGM','Net Passes',
    'Contested 3s','FG3A','OREB_CONTEST','OREB_UNCONTEST','SelfOReb',"AFGM",'Travels','DRIVE_FGA','DRIVES'
]

for stat in offensive_stats:
    if stat in data.columns:
        data[f"{stat}_per100_off"] = (data[stat] / data['OffPoss']) * 100

# Convert stats to per 100 defensive possessions
defensive_stats = [
    'DFGA_rim_defense', 'Steals',
    'Deflections', 'ChargesDrawn', 'Contested2PT Shots','BLK','DREB','PF','Contested3PT Shots',
    'Offensive Fouls Drawn','DREB_CONTEST','DREB_UNCONTEST','RecoveredBlocks','Loose BallsRecovered','OpponentPoints']

for stat in defensive_stats:
    if stat in data.columns:
        data[f"{stat}_per100_def"] = (data[stat] / data['DefPoss']) * 100

infinite_stats = data.columns[data.isin([np.inf, -np.inf]).any()]

if len(infinite_stats) > 0:
    print("Columns with infinite values detected:")
    print(infinite_stats)

    # Handle infinite values (e.g., replace with NaN or a specific value)
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("Replaced infinite values with NaN.")
else:
    print("No infinite values detected.")
data["3PtP"] = (
    (2 / (1 + np.exp(-data["FG3A_per100_off"])) - 1) * data["Fg3Pct"] 
)
data["Creation"] = (
    data["Assists_per100_off"] * 0.1843
    + (data["Points_per100_off"] + data["Turnovers_per100_off"]) * 0.0969
    - 2.3021 * data["3PtP"]
    + 0.0582
    * (
        data["Assists_per100_off"]
        * (data["Points_per100_off"] + data["Turnovers_per100_off"])
        * data["3PtP"]
    )
    - 1.1942
)
data["Load"] = (
    (data["Assists_per100_off"] - (0.38 * data["Creation"]) * 0.75)
    + data["FGA_per100_off"]
    + data["FTA_per100_off"] * 0.44
    + data["Creation"]
    + data["Turnovers_per100_off"]
)

data["cTOV"] = data["Turnovers_per100_off"] / data["Load"]
# Rim Points Saved (DFGA * DFPerc Diff * 2)
data["fTOV"] = data["Steals_per100_def"]+data["ChargesDrawn_per100_def"]+data["Offensive Fouls Drawn_per100_def"]
data["STOP%"] = data["fTOV"]+ data['RecoveredBlocks_per100_def']

data['Min'] = data['Min'].fillna(0)
data["STOP%"] = data["STOP%"].fillna(0)
data['Diff%'] = data['Diff%'].fillna(0)

# Calculate the weighted average of 'STOP%' by year, using 'Min' as weights and find rSTOP%
league_avg_stop = data.groupby('year').apply(
    lambda x: np.average(x['STOP%'], weights=x['Min']) if x['Min'].sum() > 0 else 0
).reset_index(name='League_Avg_STOP')
data = data.merge(league_avg_stop, on='year', how='left')
data["rSTOP%"] = data["STOP%"]- data["League_Avg_STOP"]

# Calculate the weighted average of 'Diff%' by year, using 'Min' as weights
league_avg_rim_defense = data.groupby('year').apply(
    lambda x: np.average(x['Diff%'], weights=x['Min']) if x['Min'].sum() > 0 else 0
).reset_index(name='League_Avg_Rim_Defense')
data = data.merge(league_avg_rim_defense, on='year', how='left')
data["rDiff%"] = data["League_Avg_Rim_Defense"] - data["Diff%"]
data["RimPointsSaved"] = data["DFGA_rim_defense_per100_def"] * (data['rDiff%']) * 2 /100


# data["G/GS%"] = data["GP"] / data["Min"]
print(data[["Player","RimPointsSaved","year",'rSTOP%']].sort_values(by="rSTOP%",ascending=False).head(10))

data.to_csv("merged_per100_dataset.csv", index=False)
print("Updated dataset with per 100 possessions stats saved as 'merged_per100_dataset.csv'")


Columns with infinite values detected:
Index(['Assist PPP', 'POT_AST_PER_MIN'], dtype='object')
Replaced infinite values with NaN.
                 Player  RimPointsSaved  year    rSTOP%
5379       RYAN ROLLINS        2.584598  2024  6.377634
4998          PAUL REED       -0.649696  2022  4.947021
4848    JEREMIAH MARTIN       -1.794510  2020  4.645680
2247       NERLENS NOEL       -1.458833  2023  4.606364
2243       NERLENS NOEL        1.139600  2019  4.217308
5437  VICTOR WEMBANYAMA        2.645756  2024  3.896501
4035       JONAH BOLDEN       -0.320189  2020  3.603426
4808   MATISSE THYBULLE       -0.122988  2021  3.451793
2244       NERLENS NOEL        1.403353  2020  3.447011
5174     ISAIAH JACKSON       -0.143125  2022  3.436706


/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/2488040631.py:103: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  league_avg_stop = data.groupby('year').apply(
/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/2488040631.py:110: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  league_avg_rim_defense = data.groupby('year').apply(


Updated dataset with per 100 possessions stats saved as 'merged_per100_dataset.csv'


In [8]:

# AssuMINg `df` is the dataset with player stats, and `rapm` is the RAPM dataset
# Step 1: Filter for years 2017-2024
year_range = range(2014, 2025)  # Include 2024
filtered_df = data[data['year'].isin(year_range)]
# Step 2: Average stats for the specified columns over the selected period
selected_columns = [
    'Player', 'AGE', 'GP', 'MIN',
    'Points_per100_off', 'Assists_per100_off', 'Turnovers_per100_off', 'FG%',
    'High Value Assist %', 'AtRimAssists_per100_off', 'TsPct',
    'AtRimFGA_per100_off', 'AtRimAccuracy',
    'LongMidRangeFGA_per100_off', 'LongMidRangeFGM_per100_off',
    'Corner3FGA_per100_off', 'Corner3FGM_per100_off', 'ThreePtAssists_per100_off',
    'DFGA_rim_defense_per100_def', 'DFG%_rim_defense', 'Diff%',
    'Deflections_per100_def','% Loose BallsRecovered DEF', 'ChargesDrawn_per100_def',
    'Contested2PT Shots_per100_def', 'on-ball-time%', 'FtPoints_per100_off','NET_RATING',
    'OREB_per100_off','OnOffRtg','BLK_per100_def','DREB_per100_def','PF_per100_def','Usage','UAFGM_per100_off','Net Passes_per100_off',
    'Contested 3s_per100_off','Steals_per100_def','FG3A_per100_off',
    'Creation','Load','cTOV','3PtP',
    'Offensive Fouls Drawn_per100_def','SelfOReb_per100_off','DREB_CONTEST_per100_def',
    'OREB_CONTEST_per100_off','OREB_UNCONTEST_per100_off','DREB_UNCONTEST_per100_def','Loose BallsRecovered_per100_def'
]



# Ensure the dataset contains the selected columns
filtered_df = filtered_df[selected_columns]

# Replace missing values with 0
filtered_df.fillna(0, inplace=True)

# Group by 'Player' and calculate the mean for numeric columns
# grouped_df = filtered_df.groupby('Player', as_index=False).mean()

# Step 3: Merge RAPM data
# Ensure 'Player' in RAPM dataset is standardized (e.g., capitalization and whitespace removal)
# rapm['Player'] = rapm['Player'].str.strip().str.upper()
# grouped_df['Player'] = grouped_df['Player'].str.strip().str.upper()

# # Merge the RAPM dataset with the grouped player stats
# smaller_df = grouped_df.merge(rapm, on='Player', how='left')  # Use 'left' join to retain all players in grouped_df

# # Check the result
# print("Merged DataFrame Preview:")
# print(smaller_df.head())

# # Save the merged dataset if needed
# smaller_df.to_csv('smaller_player_stats_with_rapm.csv', index=False)
# print("Merged dataset saved as 'smaller_player_stats_with_rapm.csv'")


In [9]:
import pandas as pd

# 1) LOAD YOUR EXISTING MERGED DATASET
merged_df = pd.read_csv("merged_per100_dataset.csv")

def format_year_range(year_start, year_end):
    """
    Convert a range of years to the format 'YYYY-YY'.
    Example: 2020, 2024 -> '2020-24'
    """
    if pd.notnull(year_start) and pd.notnull(year_end):
        return f"{year_start}-{str(year_end)[-2:]}"
    return None

# ------------------------------------------------------------------------------
# 2) FILTER FOR THE 5-YEAR WINDOW (ADJUST TO YOUR NEEDS)
#    If you truly want 2020–2024 only, change 2014 to 2020.
# ------------------------------------------------------------------------------
merged_df['year'] = pd.to_numeric(merged_df['year'], errors='coerce')
df_5yr = merged_df[merged_df['year'].between(2017, 2024)]  # or 2014, 2024 if you want that

# ------------------------------------------------------------------------------
# 3) GROUPBY TO COMPUTE AVERAGES AND YEAR MIN/MAX
# ------------------------------------------------------------------------------
# Sort by player & year if you do actual rolling; otherwise not strictly needed

# List of numeric columns you want to average
stats_cols = [
    'AGE', 'GP', 'Min',
    'Points_per100_off', 'Assists_per100_off', 'Turnovers_per100_off', 'FG%',
    'High Value Assist %', 'AtRimAssists_per100_off',
    'AtRimFGA_per100_off', 'AtRimAccuracy',
    'LongMidRangeFGA_per100_off', 'LongMidRangeFGM_per100_off',
    'Corner3FGA_per100_off', 'Corner3FGM_per100_off', 'ThreePtAssists_per100_off',
    'DFGA_rim_defense_per100_def', 'DFG%_rim_defense', 'Diff%',
    'Deflections_per100_def', '% Loose BallsRecovered DEF', 'ChargesDrawn_per100_def',
    'Contested2PT Shots_per100_def', 'on-ball-time%', 'FtPoints_per100_off',
    'NET_RATING', 'OREB_per100_off', 'OnOffRtg', 'BLK_per100_def', 'DREB_per100_def',
    'PF_per100_def', 'Usage', 'UAFGM_per100_off', 'Net Passes_per100_off',
    'Contested 3s_per100_off', 'Steals_per100_def', 'FG3A_per100_off',
    'Creation','Load','cTOV','3PtP','RimPointsSaved'
]

numeric_cols = df_5yr.select_dtypes(include=["number"]).columns
print(numeric_cols)
df_5yr.sort_values(by=["Player", "year"], inplace=True)
grouped_stats = df_5yr.groupby(["Player"])[numeric_cols].apply(
    lambda x: x.rolling(window=5,on="year", min_periods=1).mean()
)
grouped_stats.reset_index(inplace=True)
grouped_stats["year"] = (
    (grouped_stats["year"] - 4).astype(str).str[:4]
    + "-"
    + grouped_stats["year"].astype(str).str[2:4]
)

# After aggregation, columns become multi-index. Rename them:
five_year_averages = grouped_stats
five_year_averages['year'] = five_year_averages['year']
# Preview

# ------------------------------------------------------------------------------
# 4) LOAD THE NEW 5-YEAR RAPM CSV
# ------------------------------------------------------------------------------
df_new_rapm = pd.read_csv("5 Year RS RAPM.csv")
df_new_rapm.rename(columns={"Name": "Player"}, inplace=True)
df_new_rapm["Player"] = df_new_rapm["Player"].str.upper().str.strip()
df_new_rapm["year"] = df_new_rapm["Season"].str.strip()

# Standardize the Player name
five_year_averages["Player"] = five_year_averages["Player"].str.upper().str.strip()

# ------------------------------------------------------------------------------
# 5) MERGE 5-YEAR AVERAGES WITH NEW 5-YEAR RAPM
# ------------------------------------------------------------------------------
df_model = five_year_averages.merge(
    df_new_rapm, 
    on=["Player", "year"], 
    how="inner"
)

# 1. CALCULATE LEAGUE AVERAGE TsPct PER SEASON
df_model['TsPct'] = df_model['TsPct'].fillna(0)
df_model['Min'] = df_model['Min'].fillna(0)
league_avg_ts = df_model.groupby('Season').apply(
    lambda x: np.average(x['TsPct'], weights=x['Min']) if x['Min'].sum() > 0 else 0
).reset_index(name='League_Avg_TsPct')

# 2. MERGE LEAGUE AVERAGES WITH PLAYER DATA
df_model = df_model.merge(league_avg_ts, on='Season', how='left')

# 3. COMPUTE rTS
df_model['rTS'] = (df_model['TsPct'] - df_model['League_Avg_TsPct'])*100

# 4. VERIFY the new column
print(df_model[['Player', 'Season', 'TsPct', 'League_Avg_TsPct', 'rTS']].sort_values(by="rTS",ascending=False).head())

# Save the merged dataset if needed
df_model.to_csv('smaller_player_stats_with_rapm.csv', index=False)
print("Merged dataset saved as 'smaller_player_stats_with_rapm.csv'")


Index(['AGE', 'GP', 'G', 'FREQ%', 'DFGM', 'DFGA', 'DFG%', 'FG%', 'DIFF%',
       'year',
       ...
       'Creation', 'Load', 'cTOV', 'fTOV', 'STOP%', 'League_Avg_STOP',
       'rSTOP%', 'League_Avg_Rim_Defense', 'rDiff%', 'RimPointsSaved'],
      dtype='object', length=655)


/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/1027857654.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_5yr.sort_values(by=["Player", "year"], inplace=True)
/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/1027857654.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  league_avg_ts = df_model.groupby('Season').apply(


                   Player   Season     TsPct  League_Avg_TsPct        rTS
1034         MICAH POTTER  2019-23  0.800000          0.567567  23.243298
660              JAY HUFF  2019-23  0.772727          0.567567  20.516025
1230  ROBERT WILLIAMS III  2018-22  0.726101          0.559852  16.624904
1231  ROBERT WILLIAMS III  2019-23  0.730881          0.567567  16.331418
1232  ROBERT WILLIAMS III  2020-24  0.732605          0.572775  15.983062
Merged dataset saved as 'smaller_player_stats_with_rapm.csv'


In [10]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import json

# Step 1: Load the merged dataset
merged_df = pd.read_csv("smaller_player_stats_with_rapm.csv")


# Step 2: Filter players with at least 10 minutes and 20 games
combined_stats = merged_df.copy()


# Rename columns to prepare for self-merge
combined_stats_renamed = combined_stats.rename(columns={
    'Player': 'Player_teammate',
    'OnOffRtg': 'OnOffRtg_teammate',
    'Min': 'Min_teammate'
})

# Merge the DataFrame with itself to get teammates
teammates_df = pd.merge(
    combined_stats,
    combined_stats_renamed,
    on=['TEAM_ID', 'year'],
    how='left',
    suffixes=('', '_teammate')
)
# Exclude the player themselves
teammates_df = teammates_df[teammates_df['Player'] != teammates_df['Player_teammate']]

# Filter teammates who have OnOffRtg greater than player's OnOffRtg and played at least 10 minutes
teammates_filtered = teammates_df[
    (teammates_df['OnOffRtg_teammate'] > teammates_df['OnOffRtg']) &
    (teammates_df['MIN_teammate'] >= 10)
]
# Calculate the difference
teammates_filtered['Difference'] = teammates_filtered['OnOffRtg_teammate'] - teammates_filtered['OnOffRtg']
# Sum the differences for each player
sum_above = teammates_filtered.groupby(['Player', 'year'])['Difference'].sum().reset_index()

# Merge 'SumAbove' back to the original DataFrame
combined_stats = pd.merge(
    combined_stats,
    sum_above,
    on=['Player', 'year'],
    how='left'
)

# Replace NaN with 0 (players with no teammates above their OnOffRtg)
combined_stats['Difference'].fillna(0, inplace=True)
combined_stats.rename(columns={'Difference': 'SumAbove'}, inplace=True)
# print combined_stats sorted by sumabove
print(combined_stats[['Player', 'year', 'OnOffRtg', 'SumAbove']].sort_values(by='SumAbove', ascending=True).head())
combined_stats.to_csv('smaller_player_stats_with_rapm.csv', index=False)

# Step 5: Prepare Features and Target
feature_cols = ['OnOffRtg', 'NET_RATING', 'BLK_per100_def', 'DREB_per100_def', 'SumAbove']
# Check for missing values


# Drop rows with missing values
combined_stats.dropna(subset=feature_cols + ['Rapm'], inplace=True)
X = combined_stats[feature_cols]
Y = combined_stats['Rapm']


# Step 6: Train and Evaluate the Linear Regression Model
model = LinearRegression()
model.fit(X, Y)

# Make predictions
Y_pred = model.predict(X)

# Calculate R² and RMSE
r2 = r2_score(Y, Y_pred)
rmse = root_mean_squared_error(Y, Y_pred)

print(f"\nLinear Regression Model Performance:")
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

# Print the model coefficients
coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\nModel Coefficients:")
print(coefficients)
intercept = model.intercept_
print(f"Intercept (β₀): {intercept}")

                Player     year    OnOffRtg  SumAbove
746       JORDAN NWORA  2019-23  110.668622       0.0
721          JOHN WALL  2019-23  110.083368       0.0
726  JONAS VALANCIUNAS  2018-22  113.296918       0.0
727  JONAS VALANCIUNAS  2019-23  113.250502       0.0
728  JONAS VALANCIUNAS  2020-24  114.737665       0.0


/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/3956672144.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  teammates_filtered['Difference'] = teammates_filtered['OnOffRtg_teammate'] - teammates_filtered['OnOffRtg']
/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/3956672144.py:51: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method


Linear Regression Model Performance:
R²: 0.4722
RMSE: 1.4812

Model Coefficients:
           Feature  Coefficient
1       NET_RATING     0.202046
3  DREB_per100_def     0.093405
0         OnOffRtg     0.083826
2   BLK_per100_def     0.059520
4         SumAbove     0.013416
Intercept (β₀): -9.523463328401062


In [11]:
tmp =  pd.read_csv("smaller_player_stats_with_rapm.csv")

# Load your existing DataFrame (if not already loaded)
# For example:
# combined_stats_clean = pd.read_csv("smaller_player_stats_with_rapm_rTS_clean.csv")

# Define the intercept and coefficients
intercept = -6.357797067568494
coefficients = {
    'OnOffRtg': 0.058647,
    'NET_RATING': 0.282983,
    'BLK_per100_def': -0.143842,
    'DREB_per100_def': 0.122480,
    'SumAbove': 0.007007
}


# Verify that all required columns are present
required_columns = list(coefficients.keys())
missing_columns = [col for col in required_columns if col not in tmp.columns]

if missing_columns:
    raise ValueError(f"The following required columns are missing in the DataFrame: {missing_columns}")

# Calculate AuPM
tmp['AuPM'] = (
    intercept +
    (coefficients['OnOffRtg'] * tmp['OnOffRtg']) +
    (coefficients['NET_RATING'] * tmp['NET_RATING']) +
    (coefficients['DREB_per100_def'] * tmp['DREB_per100_def']) +
    (coefficients['SumAbove'] * tmp['SumAbove']) +
    (coefficients['BLK_per100_def'] * tmp['BLK_per100_def'])
)

# Display the updated DataFrame
print("\nDataFrame with 'AuPM':")
# top aupm players
print(tmp[['Player', 'year', 'AuPM']].sort_values(by='AuPM', ascending=False).head())
# Optionally, save the updated DataFrame to a new CSV file
tmp.to_csv('smaller_player_stats_with_rapm.csv', index=False)
print("\nUpdated DataFrame saved as 'combined_stats_with_AuPM.csv'")



DataFrame with 'AuPM':
            Player     year       AuPM
751   JORDAN WALSH  2020-24  10.338848
13    ADAMA SANOGO  2020-24   8.295632
454      GABE YORK  2019-23   6.962342
179   CARLIK JONES  2019-23   6.777403
535  ISAIAH MOBLEY  2019-23   6.631530

Updated DataFrame saved as 'combined_stats_with_AuPM.csv'


In [17]:

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
# Load the merged dataset
smaller_df = pd.read_csv("smaller_player_stats_with_rapm.csv")
smaller_df['Def'] = -smaller_df['Def']
# Select features and target variable
features = ["rTS","Points_per100_off","FtPoints_per100_off","AFGM_per100_off",'DRIVES_per100_off',
            "3PtP", 'Contested 3s_per100_off','FG3A_per100_off',
             "ThreePtAssists_per100_off", "cTOV",'AtRimAssists_per100_off','Net Passes_per100_off',"on-ball-time%",
             "DREB_CONTEST_per100_def","OREB_CONTEST_per100_off","DREB_UNCONTEST_per100_def","OREB_UNCONTEST_per100_off",'SelfOReb_per100_off',
             "rSTOP%","RimPointsSaved","PF_per100_def",'Contested3PT Shots_per100_def','Contested2PT Shots_per100_def',
             'Loose BallsRecovered_per100_def','Deflections_per100_def',
             'RecoveredBlocks_per100_def','Steals_per100_def','DFGA_rim_defense_per100_def', 'ChargesDrawn_per100_def']
# save features as pickle
with open ('model_features.pkl','wb') as f:
    pickle.dump(features,f)
features = ['BLK_per100_def']
target = 'Def'
smaller_df.dropna(subset=features + [target], inplace=True)
# smaller_df = smaller_df[(smaller_df['Min'] >= 10)]
smaller_df['Weight'] = smaller_df['MIN']
# split assist into each shot category
# split afgm into each shot category
# add notion fo shot location
# add scoring to and other dumb to
# Nonshooting defensive fouls drawn 
# Opponents’ field goals made and attempted
# add loose balls recovered
# Field Goals Missed Against, and added a small effect where (+0.25*blocks
# Split data into features (X) and target (y)
scaler = StandardScaler()
X = smaller_df[features]
y = smaller_df[target]
weights = smaller_df['Weight']
# Split the data into train (60%), validation (20%), and test (20%)
X_train, X_temp, y_train, y_temp, w_train, w_temp = train_test_split(
    X, y, weights, test_size=0.4, random_state=42
)

X_test, X_val, y_test, y_val, w_test, w_val = train_test_split(
    X_temp, y_temp, w_temp, test_size=0.5, random_state=42
)
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Train a linear regression model
model = LinearRegression()

model.fit(X_train, y_train,sample_weight=w_train)


# model.fit(X_train,y_train,sample_weight=w_train)
# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_val_pred = model.predict(X_val)


# Evaluate the model
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
val_r2 = r2_score(y_val, y_val_pred)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)

# Model performance
print("Model Performance:")
print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Validation R²: {val_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print("10-Fold RMSE:", -cv_scores.mean())
coefficients = pd.DataFrame({
    'Feature': model.feature_names_in_,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', ascending=False)
print("\nModel Coefficients:")
print(coefficients)
smaller_df['SPM'] = model.predict(X)

# Print the 15 highest players by SPM
top_15_spm = smaller_df[['Player','SPM',target, 'Weight','year']].sort_values(by='SPM', ascending=False).head(30)
# top_15_spm = smaller_df[['Player', 'SPM']].sort_values(by='SPM', ascending=False).head(15)
print("\nTop 15 Players by SPM:")
print(top_15_spm)

# Assume `model` is your trained LinearRegression model
coefficients = dict(zip(model.feature_names_in_, model.coef_))
intercept = model.intercept_

# Save to a JSON file
with open("model_coefficients.json", "w") as file:
    json.dump({"coefficients": coefficients, "intercept": intercept}, file)

# Add SPM (predicted RAPM) to the dataset


# Print the 15 highest differences between SPM and RAPM
smaller_df['Difference'] = abs((smaller_df['SPM'] - smaller_df[target])/smaller_df[target])
top_15_differences = smaller_df[['Player', 'Difference']].sort_values(by='Difference', ascending=False).head(30)
print("\nTop 30 Differences Between SPM and RAPM:")
print(top_15_differences)

# # Plot SPM vs RAPM
# plt.figure(figsize=(10, 6))
# plt.scatter(smaller_df[target], smaller_df['SPM'], alpha=0.6)
# plt.plot([smaller_df[target].min(), smaller_df[target].max()],
#          [smaller_df[target].min(), smaller_df[target].max()], color='red', linestyle='--')
# plt.xlabel(target)
# plt.ylabel('SPM')
# plt.title('SPM vs RAPM')
# plt.grid()
# plt.show()
# print(smaller_df[["Player",'year',target]].sort_values(by=target))

Model Performance:
Train R²: 0.0599
Test R²: -0.0280
Validation R²: 0.0554
Train RMSE: 1.1719
Test RMSE: 1.3177
Validation RMSE: 1.2946
10-Fold RMSE: 1.2106813354764567

Model Coefficients:
          Feature  Coefficient
0  BLK_per100_def     0.549211

Top 15 Players by SPM:
                   Player       SPM       Def       Weight     year
1080     MOUHAMADOU GUEYE  3.456457  0.150555   119.941667  2020-24
1440    VICTOR WEMBANYAMA  2.678281  2.262526  2106.235000  2020-24
186        CHARLES BASSEY  2.296638 -0.130348   167.860000  2018-22
1449       WALKER KESSLER  2.237283  0.076976  1598.156667  2020-24
1448       WALKER KESSLER  2.193575  0.456914  1703.206667  2019-23
1230  ROBERT WILLIAMS III  2.175857 -0.020432   865.049167  2018-22
526        ISAIAH JACKSON  2.049715 -0.960185   541.045000  2018-22
1231  ROBERT WILLIAMS III  1.958660  1.446230   856.749333  2019-23
1085         MYLES TURNER  1.939028  2.594769  1691.975333  2019-23
527        ISAIAH JACKSON  1.936775 -0.94082

In [13]:

# ------------------------------------------------------------------
# 1) LOAD YOUR merged_per100 DATASET
# ------------------------------------------------------------------
df = pd.read_csv("merged_per100_dataset.csv")
# ------------------------------------------------------------------
# 2) CALCULATE rTS
#     a) First compute league average TS% by season
#     b) Then rTS = TsPct - league_avg_ts
# ------------------------------------------------------------------
# # Ensure 'year' is numeric
# df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Drop rows with missing or invalid year
df = df.dropna(subset=['year'])

# For consistency, rename year column -> 'Season' (if you prefer “year” that’s also fine).
df.rename(columns={'year': 'Season'}, inplace=True)

# 2a) Compute league-average TS% by season
df['TsPct'] = df['TsPct'].fillna(0)
df['Min'] = df['Min'].fillna(0)
league_avg_ts = df.groupby('Season').apply(
    lambda x: np.average(x['TsPct'], weights=x['Min']) if x['Min'].sum() > 0 else 0
).reset_index(name='League_Avg_TsPct')


# 2b) Merge league averages back onto df
df = df.merge(league_avg_ts, on="Season", how="left")

# 2c) Compute rTS
df["rTS"] = (df["TsPct"] - df["League_Avg_TsPct"])*100

# ------------------------------------------------------------------
# 3) COMPUTE AuPM
# ------------------------------------------------------------------
# For demonstration, we’ll replicate the approach of:
#  - Summing the difference in OnOffRtg for all teammates with OnOffRtg > player's OnOffRtg
#  - Then building a simple regression to get the formula for AuPM
#
# If you already have a “TEAM_ID” or “Team” plus “Season” to define teammates,
# then we can do a self-merge. If not, we’ll do a simplified version where
# we assume everyone is a teammate (which obviously isn't correct, but
# it shows the structure).  

# ------------ 3a) If you do have Team + Season: ------------
#     df['Team'] = df['Team'].fillna("FA")      # or if needed
#     # rename “TEAM_ID” or “Team” in your data
# ------------------------------------------------------------

# For now, let's do a simplistic approach:  
#   We'll treat the entire league as "teammates" (which is obviously not correct),
#   just so you can see how to sum differences. 
#   Replace 'TEAM_ID' below if you have a real team ID column in your data.

df['TEAM_ID'] = df['TEAM_ID'].fillna(0) if 'TEAM_ID' in df.columns else 0

# Keep a copy
base_cols = df.columns.tolist()

# 3b) Create a self-merge to find “teammates” with higher OnOffRtg
df_renamed = df.rename(
    columns={
        'Player': 'Player_teammate',
        'OnOffRtg': 'OnOffRtg_teammate',
        'Min': 'Min_teammate',  
        # If you have multiple “min” columns, rename accordingly
    }
)

teammates_df = pd.merge(
    df,
    df_renamed,
    on=['TEAM_ID','Season'],
    suffixes=('', '_teammate'),
    how='left'
)

# Exclude the player themselves
teammates_df = teammates_df[
    teammates_df['Player'] != teammates_df['Player_teammate']
]

# Filter to only those with OnOffRtg_teammate > OnOffRtg
teammates_df = teammates_df[
    (teammates_df['OnOffRtg_teammate'] > teammates_df['OnOffRtg']) 
    & (teammates_df['Min_teammate'] >= 10)  # if you have a min filter
]

# The difference
teammates_df['Difference'] = (
    teammates_df['OnOffRtg_teammate'] - teammates_df['OnOffRtg']
)

# Sum difference for each player & season
sum_above = (
    teammates_df.groupby(['Player','Season'])['Difference']
    .sum()
    .reset_index()
    .rename(columns={'Difference':'SumAbove'})
)

# Merge SumAbove into the original df
df = df.merge(sum_above, on=['Player','Season'], how='left')
# Fill NaN with 0
df['SumAbove'] = df['SumAbove'].fillna(0)

# 3c) Now we have SumAbove for each player. We’ll fit a simple model:
#       RAPM ~ OnOffRtg + NET_RATING + BLK_per100_def + DREB_per100_def + SumAbove
#    Then use the resulting coefficients as your AuPM formula.
# 
#   NOTE: Below we assume your data has a column named 'Rapm' for each row.
#         Make sure you do have that. If not, you’ll need to ensure you merge
#         or load it into df.

required_cols = ['OnOffRtg', 'NET_RATING', 'BLK_per100_def', 'DREB_per100_def', 'SumAbove']
for col in required_cols:
    if col not in df.columns:
        print(f"Warning: Missing column {col} in df. Make sure you have it.")
        
intercept = -6.357797067568494
coef_dict = {
    'OnOffRtg': 0.058647,
    'NET_RATING': 0.282983,
    'BLK_per100_def': -0.143842,
    'DREB_per100_def': 0.122480,
    'SumAbove': 0.007007
}

# 3d) Calculate AuPM for each player
df['AuPM'] = (
    intercept
    + coef_dict['OnOffRtg']       * df['OnOffRtg'].fillna(0)
    + coef_dict['NET_RATING']     * df['NET_RATING'].fillna(0)
    + coef_dict['BLK_per100_def'] * df['BLK_per100_def'].fillna(0)
    + coef_dict['DREB_per100_def']* df['DREB_per100_def'].fillna(0)
    + coef_dict['SumAbove']       * df['SumAbove'].fillna(0)
)

# ------------------------------------------------------------------
# 4) SAVE THE RESULTS
# ------------------------------------------------------------------
# At this point, df has all your original columns + [‘rTS’, ‘SumAbove’, ‘AuPM’].
df.to_csv("merged_per100_with_rTS_AuPM.csv", index=False)

# Quick check of top 10 by AuPM
# filter low minute players
df = df[df['Min'] >= 100]



/var/folders/sb/pjwg4kzs79l5hdvl5s32yhf40000gn/T/ipykernel_27010/2752679304.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  league_avg_ts = df.groupby('Season').apply(


In [14]:

# Load the merged dataset
merged_df = pd.read_csv("merged_per100_with_rTS_AuPM.csv")
with open("model_coefficients.json", "r") as file:
    model_data = json.load(file)
merged_df = merged_df[merged_df['Season']>=2017]
manual_coefficients = model_data["coefficients"]
manual_intercept = model_data["intercept"]
manual_features = features
manual_coeff_series = pd.Series(manual_coefficients)

# inference_data_scaled = scaler.transform(inference_data)

# Align the coefficients with the DataFrame columns
# This ensures that the multiplication aligns correctly
manual_coeff_series = manual_coeff_series.reindex(manual_features)

# Calculate AuPM manually (ensure all features are numeric)
merged_df[manual_features] = merged_df[manual_features].astype(float)

merged_df['SPM_manual'] = merged_df[manual_features].dot(manual_coeff_series) + manual_intercept


add = "_O" if target == "Off" else ''
# 23. Save the updated DataFrame with SPM_manual if needed
merged_df.to_csv(f'smaller_player_stats_with_SPM{add}.csv', index=False)
print("\nUpdated DataFrame with SPM_manual saved as 'smaller_player_stats_with_SPM.csv'")

# inspect features
a = merged_df[manual_features]
# print(a.head())
# choose a year
# 21. Print top 15 players by SPM and SPM_manual
# merged_df = merged_df[merged_df['Season'] == 2024]
top_15_spm_manual = merged_df[['Player', 'SPM_manual', 'Season']].sort_values(by= 'SPM_manual', ascending=False).head(60)


print("\nTop 15 Players by SPM_manual:")
print(top_15_spm_manual)
tm = manual_features+['SPM_manual','Season']
# # print a specific player
aa = (merged_df[merged_df['Player']=='MIKE CONLEY'][tm])

print(merged_df[merged_df['Player']=='MIKE CONLEY'][['Player','SPM_manual','AuPM','rTS','Min','Season']])





Updated DataFrame with SPM_manual saved as 'smaller_player_stats_with_SPM.csv'

Top 15 Players by SPM_manual:
                     Player  SPM_manual  Season
1067           JAMES HARDEN    9.115365    2019
1065           JAMES HARDEN    8.880700    2017
2938            JOEL EMBIID    7.797806    2017
1066           JAMES HARDEN    7.734996    2018
2945            JOEL EMBIID    7.686988    2024
1068           JAMES HARDEN    7.676413    2020
2554  GIANNIS ANTETOKOUNMPO    7.443204    2020
3021           NIKOLA JOKIC    7.344723    2022
2069         DRAYMOND GREEN    7.273205    2017
171            LEBRON JAMES    7.186653    2018
3022           NIKOLA JOKIC    7.086181    2023
1345       DEMARCUS COUSINS    7.081200    2017
633            KEVIN DURANT    6.975571    2017
2553  GIANNIS ANTETOKOUNMPO    6.904020    2019
2556  GIANNIS ANTETOKOUNMPO    6.811691    2022
383              CHRIS PAUL    6.803593    2018
636            KEVIN DURANT    6.656325    2021
829       RUSSELL WESTBRO

In [15]:
# View highest rim points saved players
print("\nTop 15 Players by Rim Points Saved:")

print(merged_df[['Player', 'DFGA_rim_defense_per100_def','rDiff%','RimPointsSaved', 'Season']].sort_values(by='RimPointsSaved', ascending=False).head(30))


Top 15 Players by Rim Points Saved:
                   Player  DFGA_rim_defense_per100_def     rDiff%  \
3773           MIKE TOBEY                     8.333333  57.953999   
3490        DEYONTA DAVIS                    12.403101  25.314449   
3968         TONY BRADLEY                     8.064516  38.004968   
4020            PJ DOZIER                     8.383234  36.319719   
2361          JEFF WITHEY                     8.641975  35.104968   
4865      VINCENT POIRIER                    13.588850  22.298821   
5097       TREVELIN QUEEN                    10.089021  28.996922   
4094         ISAIAH HICKS                    10.769231  26.414449   
3955   ISAIAH HARTENSTEIN                    10.864745  23.614449   
4105          JACOB WILEY                     6.944444  36.204968   
3520        BRICE JOHNSON                    26.315789   9.553999   
3394        ALAN WILLIAMS                    13.043478  18.904968   
2938          JOEL EMBIID                    12.436869  19.753999 